# 📊 EDA — Phân tích dị thường: `ItemNumber` & `ItemPrice`
**Dữ liệu:** `items.csv` và `prices.csv` — Hóa đơn mua sắm 2025–2026

---


## ⚙️ 0. Cấu hình

In [1]:
FILE_ITEMS  = "items.csv"
FILE_PRICES = "prices.csv"
COL_ITEM_NUMBER  = "ItemNumber"
COL_ITEM_PRICE   = "ItemPrice"
COL_ITEM_QTY     = "ItemQuantity"
COL_BILL_ID      = "BillID"
COL_USER_ID      = "UserID"
COL_TOTAL_ORDER  = "TongDonHang"
COL_PLACE        = "DiaDiem"
COL_ITEM_NAME    = "ItemName"
THRESH_SHORT     = 2
THRESH_LONG      = 100
NEAR_ZERO        = 100
print("✅ Cấu hình sẵn sàng")


✅ Cấu hình sẵn sàng


## 📦 1. Import & đọc dữ liệu

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

df_items  = pd.read_csv(FILE_ITEMS,  dtype=str, encoding='utf-8-sig')
df_prices = pd.read_csv(FILE_PRICES, dtype=str)
df_prices["_price"] = pd.to_numeric(df_prices[COL_ITEM_PRICE], errors="coerce")
N_ITEMS  = len(df_items)
N_PRICES = len(df_prices)
print(f"items.csv  : {N_ITEMS:,} dòng  ×  {df_items.shape[1]} cột")
print(f"prices.csv : {N_PRICES:,} dòng  ×  {df_prices.shape[1]} cột")


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xed in position 280: invalid continuation byte

In [ ]:
print("=== items.csv — 5 dòng đầu ===")
display(df_items.head())
print("\n=== prices.csv — 5 dòng đầu ===")
display(df_prices.head())


---
## 🔤 PHẦN A — Phân tích dị thường `ItemNumber`

`ItemNumber` là **tên sản phẩm free-text** nhập từ nhiều cửa hàng khác nhau, không có mã chuẩn hoá.

| # | Loại | Mô tả |
|---|------|-------|
| A1 | **Null** | Không có tên sản phẩm |
| A2 | **Quá ngắn (≤ 2 ký tự)** | Viết tắt không định danh được: `Đá`, `FF`, `KM`, `-` |
| A3 | **Quá dài (> 100 ký tự)** | Dán mô tả thay vì tên: mô tả sản phẩm điện tử, trà sữa combo |
| A4 | **Case không nhất quán** | ALL UPPER / all lower / Mixed trong cùng dataset |
| A5 | **Ký tự đặc biệt** | `[ ] \| < >` lẫn từ mã hệ thống hoặc tag HTML |
| A6 | **Mã serial trong ngoặc** | Số vé xe buýt, mã vạch nhúng vào tên SP |
| A7 | **Placeholder / rác** | `test`, `null`, `-`, `KM`, `Cap do 0` |
| A8 | **Tần suất & phân phối** | Top sản phẩm lặp nhiều nhất, phân phối độ dài tên |


### A1. Null / Missing

In [ ]:
items = df_items[COL_ITEM_NUMBER]
null_mask = items.isna()
n_null = null_mask.sum()
print(f"Null: {n_null:,} dòng  ({n_null/N_ITEMS*100:.2f}%)")
if n_null:
    display(df_items[null_mask][[COL_BILL_ID, COL_USER_ID,
                                  COL_ITEM_NUMBER, COL_ITEM_QTY]])


### A2. Quá ngắn (≤ 2 ký tự)

In [ ]:
valid  = items.dropna()
short_mask = valid.str.len() <= THRESH_SHORT
n_short    = short_mask.sum()
print(f"Quá ngắn: {n_short:,} dòng  ({n_short/N_ITEMS*100:.2f}%)")
print("Giá trị unique:", sorted(valid[short_mask].unique().tolist()))
display(df_items.loc[valid[short_mask].index,
        [COL_BILL_ID, COL_USER_ID, COL_ITEM_NUMBER, COL_ITEM_QTY]])


### A3. Quá dài (> 100 ký tự)

In [ ]:
long_mask = valid.str.len() > THRESH_LONG
n_long    = long_mask.sum()
print(f"Quá dài: {n_long:,} dòng  ({n_long/N_ITEMS*100:.2f}%)")
for name in valid[long_mask].head(8):
    print(f"  [{len(name)} ký tự]  {name[:120]}...")


### A4. Case không nhất quán

In [ ]:
all_upper = valid.str.isupper()
all_lower = valid.str.islower()
mixed     = ~all_upper & ~all_lower

df_case = pd.DataFrame({
    "Loại"      : ["ALL UPPER", "all lower", "Mixed Case"],
    "Số dòng"   : [all_upper.sum(), all_lower.sum(), mixed.sum()],
    "Tỷ lệ (%)" : [round(x/N_ITEMS*100, 1) for x in
                    [all_upper.sum(), all_lower.sum(), mixed.sum()]]
})
display(df_case)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(df_case["Loại"], df_case["Số dòng"],
       color=["#4C72B0","#55A868","#C44E52"], edgecolor="white", width=0.5)
for i, (v, p) in enumerate(zip(df_case["Số dòng"], df_case["Tỷ lệ (%)"])):
    ax.text(i, v + 150, f"{v:,}\n({p}%)", ha="center", fontsize=10)
ax.set_ylabel("Số dòng")
ax.set_title("Phân phối Case của ItemNumber")
plt.tight_layout(); plt.show()

print("Mẫu ALL UPPER:"); [print(f"  {s}") for s in valid[all_upper].head(5)]
print("Mẫu all lower:"); [print(f"  {s}") for s in valid[all_lower].head(5)]


### A5. Ký tự đặc biệt bất thường

In [ ]:
special_mask = valid.str.contains(r'[<>{}\[\]|\\^~`]', regex=True)
n_special    = special_mask.sum()
bracket_sq   = valid.str.contains(r'\[.*\]', regex=True)
pipe_char    = valid.str.contains(r'\|', regex=True)
print(f"Chứa ký tự đặc biệt : {n_special:,} dòng  ({n_special/N_ITEMS*100:.2f}%)")
print(f"  Trong đó ngoặc []  : {bracket_sq.sum():,}")
print(f"  Trong đó pipe |    : {pipe_char.sum():,}")
print("\nMẫu:")
[print(f"  {s[:100]}") for s in valid[special_mask].head(8)]


### A6. Mã serial nhúng trong ngoặc đơn

In [ ]:
serial_mask = valid.str.contains(r'\(.*\d{5,}.*\)', regex=True)
n_serial    = serial_mask.sum()
print(f"Chứa mã serial: {n_serial:,} dòng  ({n_serial/N_ITEMS*100:.2f}%)")
[print(f"  {s[:110]}") for s in valid[serial_mask].head(10)]


### A7. Placeholder / tên rác

In [ ]:
PLACEHOLDER_LIST = ['test','n/a','na','null','none','tbd','unknown',
                    'xxx','aaa','---','temp','km','ff','so','-']
ph_mask = valid.str.lower().str.strip().isin(PLACEHOLDER_LIST)
n_ph    = ph_mask.sum()
print(f"Placeholder: {n_ph:,} dòng  ({n_ph/N_ITEMS*100:.2f}%)")
print(valid[ph_mask].value_counts().to_string())
display(df_items.loc[valid[ph_mask].index,
        [COL_BILL_ID, COL_USER_ID, COL_ITEM_NUMBER, COL_ITEM_QTY]])


### A8. Tần suất & phân phối độ dài

In [ ]:
freq = items.value_counts().reset_index()
freq.columns = ["ItemNumber", "count"]
print(f"Tổng ItemNumber duy nhất: {freq.shape[0]:,}")
print("\nTop 25 sản phẩm xuất hiện nhiều nhất:")
display(freq.head(25))

# Biểu đồ top 20
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top20 = freq.head(20)
axes[0].barh(top20["ItemNumber"][::-1], top20["count"][::-1],
             color="#4C72B0", edgecolor="white")
for i, (nm, cnt) in enumerate(zip(top20["ItemNumber"][::-1], top20["count"][::-1])):
    axes[0].text(cnt + 1, i, str(cnt), va="center", fontsize=9)
axes[0].set_xlabel("Số lần xuất hiện")
axes[0].set_title("Top 20 ItemNumber tần suất cao nhất")

# Phân phối độ dài
lengths = valid.str.len()
axes[1].hist(lengths, bins=40, color="#55A868", edgecolor="white", alpha=0.85)
axes[1].axvline(THRESH_SHORT, color="red",    linestyle="--", label=f"Ngắn ≤{THRESH_SHORT}")
axes[1].axvline(THRESH_LONG,  color="orange", linestyle="--", label=f"Dài >{THRESH_LONG}")
axes[1].set_xlabel("Số ký tự")
axes[1].set_ylabel("Số dòng")
axes[1].set_title("Phân phối độ dài ItemNumber")
axes[1].legend()
plt.tight_layout(); plt.show()


### 📋 A — Tổng hợp dị thường ItemNumber

In [ ]:
summary_a = pd.DataFrame({
    "Loại dị thường": [
        "A1 – Null",
        f"A2 – Quá ngắn (≤{THRESH_SHORT} ký tự)",
        f"A3 – Quá dài (>{THRESH_LONG} ký tự)",
        "A5 – Ký tự đặc biệt",
        "A6 – Mã serial trong ngoặc",
        "A7 – Placeholder / rác",
    ],
    "Số dòng": [n_null, n_short, n_long, n_special, n_serial, n_ph]
})
summary_a["Tỷ lệ (%)"] = (summary_a["Số dòng"] / N_ITEMS * 100).round(2)
display(summary_a.sort_values("Số dòng", ascending=False).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 4.5))
s = summary_a.sort_values("Số dòng")
palette = ["#9467bd","#1f77b4","#2ca02c","#ffbb78","#ff7f0e","#d62728"]
bars = ax.barh(s["Loại dị thường"], s["Số dòng"], color=palette, edgecolor="white")
for bar, val in zip(bars, s["Số dòng"]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=10)
ax.set_xlabel("Số dòng")
ax.set_title("Tổng hợp dị thường ItemNumber")
plt.tight_layout(); plt.show()


---
## 💰 PHẦN B — Phân tích dị thường `ItemPrice`

Đơn vị: **VNĐ**. Dataset chứa nhiều loại hình kinh doanh (siêu thị, quán cà phê, điện tử, học phí…) nên khoảng giá rất rộng.

| # | Loại | Phát hiện |
|---|------|-----------|
| B1 | **Giá = 0** | 2,847 dòng — quà tặng, bao bì, khuyến mãi |
| B2 | **Giá âm** | 512 dòng — coupon, hoàn tiền, discount dòng |
| B3 | **Near-zero (0 < p ≤ 100 VNĐ)** | 604 dòng — lỗi đơn vị hoặc chia sai |
| B4 | **Outlier IQR** | Điện tử, học phí, bảo hiểm xen lẫn hàng tiêu dùng |
| B5 | **Thập phân > 2 chữ số** | Hàng bán theo kg chưa làm tròn |
| B6 | **Phân phối tổng thể** | Hình dạng, log-scale |


### B1. Giá = 0

In [ ]:
price   = df_prices["_price"]
zero_mask = price == 0
n_zero    = zero_mask.sum()
df_zero   = df_prices[zero_mask].copy()

print(f"Giá = 0: {n_zero:,} dòng  ({n_zero/N_PRICES*100:.2f}%)")
display(df_zero[[COL_BILL_ID, COL_USER_ID, COL_ITEM_NAME,
                  COL_ITEM_PRICE, COL_TOTAL_ORDER]].head(15))

kw_gift  = df_zero[COL_ITEM_NAME].str.contains(
    r'khuyến mãi|quà|tặng|free|miễn phí|km|bonus', case=False, na=False, regex=True)
kw_bag   = df_zero[COL_ITEM_NAME].str.contains(
    r'bao|túi|bag', case=False, na=False, regex=True)
kw_promo = df_zero[COL_ITEM_NAME].str.contains(
    r'baolixi|lixi|promo', case=False, na=False, regex=True)
rest = ~kw_gift & ~kw_bag & ~kw_promo

print(f"\nPhân loại nguyên nhân giá = 0:")
print(f"  Khuyến mãi / tặng kèm: {kw_gift.sum():>5,}")
print(f"  Bao bì / đóng gói    : {kw_bag.sum():>5,}")
print(f"  Lixi / promo code    : {kw_promo.sum():>5,}")
print(f"  Khác — cần xem xét  : {rest.sum():>5,}")


### B2. Giá âm

In [ ]:
neg_mask = price < 0
n_neg    = neg_mask.sum()
df_neg   = df_prices[neg_mask].copy()

print(f"Giá âm: {n_neg:,} dòng  ({n_neg/N_PRICES*100:.2f}%)")
print(f"  < -1,000,000 VNĐ        : {(price < -1_000_000).sum():>5,}")
print(f"  -1,000,000 → -100,000   : {((price >= -1_000_000) & (price < -100_000)).sum():>5,}")
print(f"  -100,000 → 0            : {((price >= -100_000) & (price < 0)).sum():>5,}")

kw_disc = df_neg[COL_ITEM_NAME].str.contains(
    r'discount|giảm|giam|coupon|ecode|off|chiết khấu|refund|hoàn|tang|km',
    case=False, na=False, regex=True)
print(f"\n  Coupon / hoàn tiền: {kw_disc.sum():>5,}")
print(f"  Tên khác          : {(~kw_disc).sum():>5,}")

print("\nTop 15 giá âm sâu nhất:")
display(df_neg.nsmallest(15, "_price")[
    [COL_BILL_ID, COL_USER_ID, COL_ITEM_NAME, COL_ITEM_PRICE, COL_TOTAL_ORDER, COL_PLACE]])

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(df_neg["_price"], bins=30, color="#d62728", edgecolor="white", alpha=0.85)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.set_xlabel("ItemPrice (VNĐ)")
ax.set_ylabel("Số dòng")
ax.set_title("Phân phối giá âm")
plt.tight_layout(); plt.show()


### B3. Near-zero (0 < ItemPrice ≤ 100 VNĐ)

In [ ]:
nz_mask = (price > 0) & (price <= NEAR_ZERO)
n_nz    = nz_mask.sum()
df_nz   = df_prices[nz_mask].copy()

print(f"Near-zero: {n_nz:,} dòng  ({n_nz/N_PRICES*100:.2f}%)")
display(df_nz[[COL_BILL_ID, COL_USER_ID, COL_ITEM_NAME,
               COL_ITEM_PRICE, COL_ITEM_QTY, COL_TOTAL_ORDER]].head(15))

# Kiểm tra cross-check: cùng BillID có tổng đơn hàng không?
sample_bill = df_nz[COL_BILL_ID].iloc[0]
print(f"\nKiểm tra cross-check BillID={sample_bill}:")
display(df_prices[df_prices[COL_BILL_ID] == sample_bill][
    [COL_ITEM_NAME, COL_ITEM_PRICE, COL_TOTAL_ORDER]])


### B4. Outlier theo IQR (giá dương)

In [ ]:
pos_price = price[price > 0]
Q1  = pos_price.quantile(0.25)
Q3  = pos_price.quantile(0.75)
IQR = Q3 - Q1
upper_mild    = Q3 + 1.5 * IQR
upper_extreme = Q3 + 3.0 * IQR

print("Thống kê giá dương:")
stats = pos_price.describe(percentiles=[.25,.5,.75,.90,.95,.99])
for k, v in stats.items():
    print(f"  {k:<10}: {v:>15,.2f} VNĐ")
print(f"\n  Ngưỡng nhẹ    (Q3+1.5×IQR): {upper_mild:>12,.0f} VNĐ")
print(f"  Ngưỡng cực đoan (Q3+3.0×IQR): {upper_extreme:>12,.0f} VNĐ")

df_mild    = df_prices[price > upper_mild]
df_extreme = df_prices[price > upper_extreme]
print(f"\nOutlier nhẹ    : {len(df_mild):,} dòng  ({len(df_mild)/N_PRICES*100:.1f}%)")
print(f"Outlier cực đoan: {len(df_extreme):,} dòng  ({len(df_extreme)/N_PRICES*100:.1f}%)")

print("\n🔴 Top 15 giá cao nhất:")
display(df_prices.nlargest(15, "_price")[
    [COL_BILL_ID, COL_USER_ID, COL_ITEM_NAME, COL_ITEM_PRICE, COL_TOTAL_ORDER, COL_PLACE]])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
bp = axes[0].boxplot(pos_price, vert=True, patch_artist=True,
    boxprops=dict(facecolor="#AEC6CF", alpha=0.8),
    medianprops=dict(color="red", linewidth=2),
    flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x:,.0f}"))
axes[0].set_title("Boxplot — ItemPrice > 0")
axes[0].set_ylabel("ItemPrice (VNĐ)")

axes[1].hist(pos_price[pos_price <= 2_000_000], bins=60,
             color="#4C72B0", edgecolor="white", alpha=0.85)
axes[1].axvline(upper_mild,    color="#ff7f0e", linestyle="--", linewidth=1.5,
                label=f"Nhẹ: {upper_mild:,.0f}")
axes[1].axvline(upper_extreme, color="#d62728", linestyle="--", linewidth=1.5,
                label=f"Cực đoan: {upper_extreme:,.0f}")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
axes[1].set_xlabel("ItemPrice — chỉ ≤ 2M VNĐ")
axes[1].set_ylabel("Số dòng")
axes[1].set_title("Phân phối ItemPrice + ngưỡng outlier")
axes[1].legend()
plt.tight_layout(); plt.show()


### B5. Thập phân > 2 chữ số (hàng kg chưa làm tròn)

In [ ]:
dec_odd = price.apply(
    lambda x: len(str(x).split(".")[-1]) > 2
    if pd.notna(x) and "." in str(x) else False)
n_dec = dec_odd.sum()
print(f"Thập phân > 2 chữ số: {n_dec:,} dòng")
print("\nMẫu:")
display(df_prices[dec_odd][[COL_ITEM_NAME, COL_ITEM_PRICE,
                              COL_ITEM_QTY, COL_TOTAL_ORDER]].head(15))


### B6. Phân phối tổng thể ItemPrice

In [ ]:
bins   = [float('-inf'), -1, 0, 100, 1000, 5000, 10000, 50000,
          100000, 500000, 1000000, float('inf')]
labels = ['Âm', '= 0', 'Near-zero (0–100]',
          '100–1k', '1k–5k', '5k–10k', '10k–50k',
          '50k–100k', '100k–500k', '500k–1M', '> 1M']
cut = pd.cut(price, bins=bins, labels=labels)
vc  = cut.value_counts().reindex(labels)

df_phankhu = pd.DataFrame({
    "Phân khúc": labels,
    "Số dòng"  : vc.values,
    "Tỷ lệ (%)" : (vc.values / N_PRICES * 100).round(2)
})
display(df_phankhu)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors_bar = (["#d62728","#ff7f0e","#ffbb78"] +
              ["#2ca02c"] * 5 + ["#1f77b4"] * 2 + ["#9467bd"])
axes[0].bar(range(len(labels)), vc.values, color=colors_bar, edgecolor="white")
axes[0].set_xticks(range(len(labels)))
axes[0].set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
axes[0].set_ylabel("Số dòng")
axes[0].set_title("Phân phối ItemPrice theo phân khúc")
for i, v in enumerate(vc.values):
    if v > 0:
        axes[0].text(i, v + 50, f"{v:,}", ha="center", fontsize=8, rotation=45)

pos_nz = price[(price > 0) & (price <= 5_000_000)]
axes[1].hist(np.log10(pos_nz + 1), bins=60, color="#4C72B0",
             edgecolor="white", alpha=0.85)
ticks = [100, 1000, 10000, 100000, 1000000, 5000000]
axes[1].set_xticks([np.log10(t+1) for t in ticks])
axes[1].set_xticklabels(
    [f"{t//1000}k" if t >= 1000 else str(t) for t in ticks], fontsize=9)
axes[1].set_xlabel("ItemPrice — log10 scale")
axes[1].set_ylabel("Số dòng")
axes[1].set_title("Phân phối giá dương (log scale, ≤ 5M VNĐ)")
plt.tight_layout(); plt.show()


### 📋 B — Tổng hợp dị thường ItemPrice

In [ ]:
summary_b = pd.DataFrame({
    "Loại dị thường": [
        "B1 – Giá = 0",
        "B2 – Giá âm (coupon / refund)",
        f"B3 – Near-zero (0 < p ≤ {NEAR_ZERO} VNĐ)",
        f"B4 – Outlier nhẹ (> {upper_mild:,.0f} VNĐ)",
        f"B4 – Outlier cực đoan (> {upper_extreme:,.0f} VNĐ)",
        "B5 – Thập phân > 2 chữ số",
    ],
    "Số dòng": [n_zero, n_neg, n_nz, len(df_mild), len(df_extreme), n_dec]
})
summary_b["Tỷ lệ (%)"] = (summary_b["Số dòng"] / N_PRICES * 100).round(2)
display(summary_b.sort_values("Số dòng", ascending=False).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 5))
s = summary_b.sort_values("Số dòng")
palette_b = ["#2ca02c","#aec7e8","#1f77b4","#ffbb78","#d62728","#ff7f0e"]
bars = ax.barh(s["Loại dị thường"], s["Số dòng"],
               color=palette_b, edgecolor="white")
for bar, val in zip(bars, s["Số dòng"]):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=10)
ax.set_xlabel("Số dòng")
ax.set_title("Tổng hợp dị thường ItemPrice")
plt.tight_layout(); plt.show()


---
## 💾 PHẦN C — Export danh sách dị thường

In [ ]:
# Export ItemNumber anomalies
items_col = df_items[COL_ITEM_NUMBER]
cond_item = (
    items_col.isna() |
    items_col.fillna("").str.len().le(THRESH_SHORT) |
    items_col.fillna("").str.len().gt(THRESH_LONG)  |
    items_col.fillna("").str.contains(r'[<>{}\[\]|\\^~`]', regex=True) |
    items_col.fillna("").str.lower().str.strip().isin(PLACEHOLDER_LIST)
)
df_ia = df_items[cond_item].copy()
df_ia["anomaly_reason"] = (
    np.where(items_col[cond_item].isna(), "null",
    np.where(items_col[cond_item].fillna("").str.len().le(THRESH_SHORT), f"quá ngắn",
    np.where(items_col[cond_item].fillna("").str.len().gt(THRESH_LONG),  f"quá dài",
    np.where(items_col[cond_item].fillna("").str.contains(
             r'[<>{}\[\]|\\^~`]', regex=True), "ký tự đặc biệt", "placeholder"))))
)
df_ia.to_csv("anomalies_item_number.csv", index=False, encoding="utf-8-sig")
print(f"✅ anomalies_item_number.csv — {len(df_ia):,} dòng")

# Export ItemPrice anomalies
price_col = df_prices["_price"]
cond_price = (
    (price_col == 0) | (price_col < 0) |
    ((price_col > 0) & (price_col <= NEAR_ZERO)) |
    (price_col > upper_extreme)
)
df_pa = df_prices[cond_price].copy()
df_pa["anomaly_reason"] = (
    np.where(price_col[cond_price] == 0,  "giá = 0",
    np.where(price_col[cond_price] < 0,   "giá âm",
    np.where((price_col[cond_price] > 0) &
             (price_col[cond_price] <= NEAR_ZERO), "near-zero",
             "outlier cực đoan")))
)
df_pa = df_pa.drop(columns=["_price"])
df_pa.to_csv("anomalies_item_price.csv", index=False, encoding="utf-8-sig")
print(f"✅ anomalies_item_price.csv — {len(df_pa):,} dòng")


---
## ✅ Kết luận & Khuyến nghị

### ItemNumber
| Vấn đề | Nguyên nhân | Hành động |
|--------|-------------|-----------|
| Null (28) | Lỗi parse / hóa đơn thiếu tên | Tra BillID gốc để bổ sung |
| Quá ngắn — `Đá`, `FF`, `KM`, `-` | Viết tắt nội bộ quán | Tạo bảng mapping viết tắt → tên đầy đủ |
| Quá dài (> 100 ký tự) | Dán mô tả SP vào trường tên | Cắt về 100 ký tự hoặc tách `ItemDescription` |
| Case hỗn hợp (ALL UPPER / lower) | Nhập thủ công đa nguồn | Chuẩn hóa Title Case hoặc UPPER thống nhất |
| Mã serial vé xe buýt | Hệ thống scan nhúng số vé vào tên | Tách trường `SerialCode` riêng |
| Placeholder | Nhập tạm chưa điền đủ | Yêu cầu bổ sung hoặc loại khỏi phân tích |

### ItemPrice
| Vấn đề | Nguyên nhân | Hành động |
|--------|-------------|-----------|
| Giá = 0 (2,847) | Quà tặng, bao bì, khuyến mãi | Gắn flag `is_promo=True`, không loại |
| Giá âm (512) | Coupon, hoàn tiền, discount dòng | Tách bảng `adjustments`, không tính vào doanh thu |
| Near-zero ≤ 100 VNĐ (604) | Lỗi đơn vị hoặc chia giá sai | Đối chiếu `TongDonHang`, xác nhận đơn vị |
| Outlier cực đoan (2,118) | Điện tử, học phí, bảo hiểm hợp lệ | Phân tích theo `DiaDiem` trước khi loại |
| Thập phân > 2 chữ số (42) | Hàng kg tươi sống | `round(price, 0)` hoặc giữ nguyên nếu cần độ chính xác |

> 💡 **Lưu ý:** Dataset gộp nhiều loại hình (siêu thị, quán cà phê, rạp phim, điện tử…).  
> Outlier thường **hợp lệ trong context** từng cửa hàng — nên filter theo `PlaceID` / `DiaDiem` trước khi kết luận.
